In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datetime import datetime
import joblib
import os
import os

# Set working directory
os.chdir("G:/ML/backend")

# -----------------------------
# TRAINING PHASE
# -----------------------------

X_list = []
y_list = []

chunk_size = 100000
chunks = pd.read_csv("C:/Users/HP/US_Accidents_March23.csv", chunksize=chunk_size)

for chunk in chunks:
    condition_counts = chunk['Weather_Condition'].value_counts()
    filtered_chunk = chunk[chunk['Weather_Condition'].isin(condition_counts[condition_counts > 100].index)]
    filtered_chunk = filtered_chunk.dropna(subset=['Temperature(F)', 'Wind_Speed(mph)', 'Severity'])

    filtered_chunk['Start_Time'] = pd.to_datetime(filtered_chunk['Start_Time'], errors='coerce')
    filtered_chunk['End_Time'] = pd.to_datetime(filtered_chunk['End_Time'], errors='coerce')

    filtered_chunk['Start_Hour'] = filtered_chunk['Start_Time'].dt.hour
    filtered_chunk['End_Hour'] = filtered_chunk['End_Time'].dt.hour

    categorical_selected = ['City', 'Weather_Condition']
    numeric_selected = ['Wind_Speed(mph)', 'Temperature(F)', 'Start_Hour', 'End_Hour']
    selected_features = categorical_selected + numeric_selected

    X_chunk = filtered_chunk[selected_features]
    y_chunk = filtered_chunk['Severity']

    X_list.append(X_chunk)
    y_list.append(y_chunk)

X = pd.concat(X_list, axis=0)
y = pd.concat(y_list, axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

label_encoders = {}
for feature in ['City', 'Weather_Condition']:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[feature], X_test[feature]]).astype(str))
    X_train_encoded[feature] = le.transform(X_train[feature].astype(str))
    X_test_encoded[feature] = le.transform(X_test[feature].astype(str))
    label_encoders[feature] = le

scaler = StandardScaler()
X_train_encoded[numeric_selected] = scaler.fit_transform(X_train_encoded[numeric_selected])
X_test_encoded[numeric_selected] = scaler.transform(X_test_encoded[numeric_selected])

# ✅ Updated model with limited size for memory efficiency
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42
)
rf_model.fit(X_train_encoded, y_train)

y_pred = rf_model.predict(X_test_encoded)
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# -----------------------------
# SAVE MODEL AND ENCODERS
# -----------------------------

# Create encoders folder if it doesn't exist
os.makedirs("encoders", exist_ok=True)

# Save using joblib
joblib.dump(rf_model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders['City'], "encoders/city_encoder.pkl")
joblib.dump(label_encoders['Weather_Condition'], "encoders/weather_encoder.pkl")


Model Accuracy: 0.8051636946254153


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00     13136
           2       0.81      1.00      0.89   1137257
           3       0.69      0.00      0.01    226815
           4       0.89      0.00      0.00     35850

    accuracy                           0.81   1413058
   macro avg       0.60      0.25      0.23   1413058
weighted avg       0.78      0.81      0.72   1413058



C:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


['encoders/weather_encoder.pkl']

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datetime import datetime

# -----------------------------
# TRAINING PHASE
# -----------------------------

X_list = []
y_list = []

chunk_size = 100000
chunks = pd.read_csv('C:/Users/HP/US_Accidents_March23.csv', chunksize=chunk_size)

for chunk in chunks:
    condition_counts = chunk['Weather_Condition'].value_counts()
    filtered_chunk = chunk[chunk['Weather_Condition'].isin(condition_counts[condition_counts > 100].index)]
    filtered_chunk = filtered_chunk.dropna(subset=['Temperature(F)', 'Wind_Speed(mph)', 'Severity'])

    # Convert Start and End Time
    filtered_chunk['Start_Time'] = pd.to_datetime(filtered_chunk['Start_Time'], errors='coerce')
    filtered_chunk['End_Time'] = pd.to_datetime(filtered_chunk['End_Time'], errors='coerce')

    # Extract hours
    filtered_chunk['Start_Hour'] = filtered_chunk['Start_Time'].dt.hour
    filtered_chunk['End_Hour'] = filtered_chunk['End_Time'].dt.hour

    # Select relevant features
    categorical_selected = ['City', 'Weather_Condition']
    numeric_selected = ['Wind_Speed(mph)', 'Temperature(F)', 'Start_Hour', 'End_Hour']
    selected_features = categorical_selected + numeric_selected

    X_chunk = filtered_chunk[selected_features]
    y_chunk = filtered_chunk['Severity']

    X_list.append(X_chunk)
    y_list.append(y_chunk)

X = pd.concat(X_list, axis=0)
y = pd.concat(y_list, axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Encode categorical features
label_encoders = {}
for feature in ['City', 'Weather_Condition']:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[feature], X_test[feature]]).astype(str))
    X_train_encoded[feature] = le.transform(X_train[feature].astype(str))
    X_test_encoded[feature] = le.transform(X_test[feature].astype(str))
    label_encoders[feature] = le

# Scale numerical features
scaler = StandardScaler()
X_train_encoded[numeric_selected] = scaler.fit_transform(X_train_encoded[numeric_selected])
X_test_encoded[numeric_selected] = scaler.transform(X_test_encoded[numeric_selected])

# Train model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_encoded, y_train)

# Evaluate model
y_pred = rf_model.predict(X_test_encoded)
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# -----------------------------
# USER INPUT PHASE
# -----------------------------

print("\n=== Predict Accident Severity ===")

city_input = input("Enter City: ")
weather_input = input("Enter Weather Condition: ")
wind_speed_input = float(input("Enter Wind Speed (mph): "))
temperature_input = float(input("Enter Temperature (F): "))
start_time_input = input("Enter Start Time (HH:MM, 24-hour format): ")
end_time_input = input("Enter End Time (HH:MM, 24-hour format): ")

try:
    start_hour = datetime.strptime(start_time_input, "%H:%M").hour
except ValueError:
    print("Invalid start time format. Using default 8:00.")
    start_hour = 8

try:
    end_hour = datetime.strptime(end_time_input, "%H:%M").hour
except ValueError:
    print("Invalid end time format. Using default 9:00.")
    end_hour = 9

user_data = pd.DataFrame([{
    'City': city_input,
    'Weather_Condition': weather_input,
    'Wind_Speed(mph)': wind_speed_input,
    'Temperature(F)': temperature_input,
    'Start_Hour': start_hour,
    'End_Hour': end_hour
}])

# Encode user input
for feature in ['City', 'Weather_Condition']:
    le = label_encoders[feature]
    if user_data[feature][0] in le.classes_:
        user_data[feature] = le.transform(user_data[feature].astype(str))
    else:
        print(f"⚠️ Warning: '{user_data[feature][0]}' not in training data for '{feature}'. Using fallback encoding.")
        user_data[feature] = le.transform([le.classes_[0]])

# Scale numeric features
user_data[numeric_selected] = scaler.transform(user_data[numeric_selected])

# Predict
user_prediction = rf_model.predict(user_data)
print("\nPredicted Accident Severity Level:", user_prediction[0])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.stats import chi2_contingency

# Load the dataset
accident_weather_dataset = pd.read_csv('trial.csv')

# Filter rows where Weather_Condition count > 100
condition_counts = accident_weather_dataset['Weather_Condition'].value_counts()
filtered_dataset = accident_weather_dataset[
    accident_weather_dataset['Weather_Condition'].isin(condition_counts[condition_counts > 100].index)
]

# Drop rows with missing values for relevant features
filtered_dataset = filtered_dataset.dropna(
    subset=['Start_Lat', 'Start_Lng', 'Weather_Condition', 'Wind_Speed(mph)', 'Severity']
)

# Encode Weather_Condition (fit and save encoder separately)
weather_encoder = LabelEncoder()
filtered_dataset['Weather_Condition'] = weather_encoder.fit_transform(filtered_dataset['Weather_Condition'])

# Identify numeric and categorical features
numeric_features = accident_weather_dataset.select_dtypes(include=['float64', 'int64']).columns
categorical_features = accident_weather_dataset.select_dtypes(include=['object']).columns

# Check correlation with Severity
corr_matrix = filtered_dataset[numeric_features].corr()
target_corr = corr_matrix['Severity'].sort_values(ascending=False)
print("Correlation with Severity:\n", target_corr)

# Chi-Square Test for each categorical feature
df_encoded = filtered_dataset.copy()
for feature in categorical_features:
    if feature in df_encoded.columns:
        le_temp = LabelEncoder()
        df_encoded[feature] = le_temp.fit_transform(df_encoded[feature])
        contingency_table = pd.crosstab(df_encoded[feature], df_encoded['Severity'])
        chi2_stat, p_val, dof, ex = chi2_contingency(contingency_table)
        print(f'Chi2 Test for {feature}: p-value = {p_val}')

# Select relevant features and target variable
X = filtered_dataset[['Start_Lat', 'Start_Lng', 'Weather_Condition', 'Wind_Speed(mph)']]
y = filtered_dataset['Severity']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numeric features
numeric_selected = ['Start_Lat', 'Start_Lng', 'Wind_Speed(mph)']
scaler = StandardScaler()
X_train[numeric_selected] = scaler.fit_transform(X_train[numeric_selected])
X_test[numeric_selected] = scaler.transform(X_test[numeric_selected])

# Train Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf_model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Predict severity for new path data
new_path_data = pd.DataFrame({
    'Start_Lat': [34.0522],
    'Start_Lng': [-118.2437],
    'Weather_Condition': ['Clear'],
    'Wind_Speed(mph)': [5]
})

# Check if the value exists in fitted encoder
if new_path_data['Weather_Condition'][0] in weather_encoder.classes_:
    new_path_data['Weather_Condition'] = weather_encoder.transform(new_path_data['Weather_Condition'])
else:
    # Assign the most frequent value or handle accordingly
    most_common_condition = filtered_dataset['Weather_Condition'].mode()[0]
    print("Unseen weather condition. Using most common encoded value instead.")
    new_path_data['Weather_Condition'] = [most_common_condition]

# Scale the numeric features
new_path_data[numeric_selected] = scaler.transform(new_path_data[numeric_selected])

# Predict and show result
predicted_severity = rf_model.predict(new_path_data)
print("Predicted Severity for New Path:", predicted_severity)